# Model inference - Walmart Store Sales Forecasting

Final deliverable: take the **completely un-preprocessed** `data/raw/test.csv`, push it
through the winning model fetched from the **MLflow Model Registry**, and write
`submissions/final_submission.csv` in the Kaggle format.

## Which model ships, and why

**Prophet** (`src.prophet_forecaster.ProphetPanelPipeline`) - one Prophet model per
`(Store, Dept)` series (~3,300 of them), with a department-level week-of-year seasonal
fallback for series too short to fit. It is the best model **on the Kaggle leaderboard**,
which is the number that decides this project.

Note that the shared expanding-window CV WMAE does *not* rank it first (DLinear scores
1,618 there). That disagreement is expected, and `model_experiment_prophet.ipynb` explains
it at length: every fold of the shared CV grid ends inside train, so **no CV fold can
contain a Thanksgiving or a Christmas week** - the two weeks WMAE weights 5x, and the two
weeks the real test horizon is largely made of. The Prophet notebook's `TESTLIKE` fold (a
39-week structural replica of the test span) is the fold that actually tracks the
leaderboard. Ranked by leaderboard score, Prophet wins.

## Why the whole thing is one `.predict()` call

`ProphetPanelPipeline.predict()` takes a **raw `test.csv`-shaped frame** and returns a plain
`np.ndarray` of predictions. It needs only `Store`, `Dept` and `Date`: it does not need
`IsHoliday`, and it does not need the `features.csv` / `stores.csv` merge at all, because
holiday effects live inside each per-series Prophet model (learned from the holidays frame
and the Christmas day-count regressors, both keyed off the date). Preprocessing, feature
engineering and inference genuinely collapse into one step here - there is no separate
transform left to run.

## Steps

1. Init (Colab-compatible), credentials, MLflow tracking store, Model Registry coordinates
2. Load the raw, untouched `data/raw/test.csv`
3. Bootstrap the registry (run-once; see that section for why it is needed)
4. Fetch the pipeline from the Model Registry
5. `pipeline.predict(raw_test)` - preprocessing + features + inference in one call
6. Format, sanity-check, and write `submissions/final_submission.csv`

**Runs on CPU.** Prophet fits through Stan and has no GPU path; the ~3,000 per-series
forecasts are parallelised across cores with joblib. A GPU runtime buys nothing here.

##  Init cell (Colab-compatible)

Byte-identical to the Prophet notebook's init cell: clone/pull the repo, install
`requirements.txt`, `chdir` into `notebooks/`, and copy `data/raw` down from Drive (the
competition data is gitignored, so it is never in the clone).

**This cell is what makes `import src...` work.** On Colab the working directory is
`/content`, not the repo, so without the `os.chdir` + `sys.path.append` below the imports
cell dies with `ModuleNotFoundError: No module named 'src'`. Locally it is a no-op.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    REPO_URL = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"
    REPO_DIR = "/content/walmart-sales-forecasting"
    BRANCH = "master"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        # The Colab clone is a DISPOSABLE working copy, so resync it hard instead of pulling.
        # A plain `git pull` fails here with a bare CalledProcessError: submissions/*.csv and
        # submissions/*.png are tracked files, every notebook run rewrites them, and git
        # refuses to merge over locally modified tracked files. Discarding whatever the last
        # run left behind is exactly what we want on a throwaway VM.
        #
        # `clean -fd` (no -x) removes untracked files but NOT gitignored ones, so data/raw/
        # copied down from Drive survives. Anything you edit inside the clone is destroyed -
        # author changes locally and push, never here.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "clean", "-fd"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         f"{REPO_DIR}/requirements.txt", "--quiet"],
        check=True,
    )

    os.chdir(f"{REPO_DIR}/notebooks")

    from google.colab import drive
    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    repo_data_dir = Path(REPO_DIR) / "data" / "raw"
    if drive_data_dir.exists():
        subprocess.run(["cp", "-r", f"{drive_data_dir}/.", str(repo_data_dir)], check=True)
    else:
        raise FileNotFoundError(
            f"Expected Drive data folder not found at {drive_data_dir}. "
            "Create it (or add it as a My Drive shortcut) before running this notebook."
        )

sys.path.append(str(Path.cwd().parent))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##  Imports and paths

In [2]:
import joblib
import numpy as np
import pandas as pd

import mlflow
from mlflow import MlflowClient


def find_repo_root() -> Path:
    """Walk up from the cwd until we find the directory that holds `src/`.

    The init cell already chdir'd into `notebooks/`, so this is normally just
    `Path.cwd().parent`. We search rather than assume, because the cwd depends on how the
    notebook is driven (Colab starts in /content, papermill/nbconvert can start anywhere),
    and guessing from the directory name is exactly what breaks on Colab.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "prophet_forecaster.py").is_file():
            return candidate
    raise RuntimeError(
        f"Could not find the repo root (no parent of {Path.cwd()} contains src/). "
        "Run the init cell above, or start the kernel from inside the repo."
    )


# The repo root has to be importable as `src.*`. This is not only for our own imports: the
# pipeline was pickled as `src.prophet_forecaster.ProphetPanelPipeline`, so that exact module
# path must resolve for joblib to reconstruct the object.
REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.prophet_forecaster import ProphetPanelPipeline  # noqa: E402,F401  (needs sys.path above)

DATA_DIR = REPO_ROOT / "data" / "raw"
SRC_DIR = REPO_ROOT / "src"
SUBMISSIONS_DIR = REPO_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(exist_ok=True)

SUBMISSION_PATH = SUBMISSIONS_DIR / "final_submission.csv"

# Every row of the Kaggle test set. Hardcoded so the shape check at the end is a real
# assertion about the submission, not a tautology derived from the file we just read.
EXPECTED_ROWS = 115064

pd.set_option("display.max_columns", 50)
print("running on Colab:", IS_COLAB)
print("repo root:       ", REPO_ROOT)

running on Colab: True
repo root:        /content/walmart-sales-forecasting


##  Manual credentials override (VS Code / non-Colab-UI runtimes)

`google.colab.userdata` (Colab Secrets) can only be read from the Colab **browser UI**. When
the Colab runtime is driven from VS Code or any other non-UI frontend it times out (`Secrets
can only be fetched when running from the Colab UI`). This cell sets the DagsHub creds
directly instead, and the credentials cell below skips `userdata` whenever these env vars are
already set.

`getpass` is used so the token is never written into this committed notebook - run the cell
and paste the values when prompted. Leave a prompt blank to fall through to Colab Secrets /
`.env` below (e.g. when you *are* on the Colab UI).

In [3]:
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token

##  Load DagsHub credentials

`MLFLOW_TRACKING_USERNAME`/`MLFLOW_TRACKING_PASSWORD` are never hardcoded in this notebook
(it gets committed to the shared repo, so a hardcoded token would leak).

- On the Colab UI: read from Colab secrets - add `DAGSHUB_USERNAME` and `DAGSHUB_TOKEN` via
  the key icon in the left sidebar, and enable "Notebook access" for both.
- From VS Code / non-UI runtimes: use the manual-override cell above.
- Locally: falls back to a gitignored `.env` in the repo root.

Same 3-tier cascade as every `model_experiment_*.ipynb`.

In [4]:
if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided (e.g. by the manual-override cell above when driving the Colab runtime
    # from VS Code, where google.colab.userdata would time out). Note: userdata.get(...) must
    # NOT be evaluated in this case - it blocks for ~minutes and raises when there is no Colab
    # browser UI to answer it.
    creds_source = "pre-set environment variables"
elif IS_COLAB:
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
    creds_source = "Colab secrets (DAGSHUB_USERNAME / DAGSHUB_TOKEN)"
else:
    from dotenv import load_dotenv

    env_path = REPO_ROOT / ".env"
    load_dotenv(env_path)
    creds_source = str(env_path)

assert os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"), (
    f"MLFLOW_TRACKING_USERNAME/PASSWORD not set (tried: {creds_source}). "
    "On the Colab UI: add DAGSHUB_USERNAME and DAGSHUB_TOKEN as Colab secrets "
    "(key icon in the left sidebar) and enable notebook access for both. "
    "From VS Code / non-UI runtimes: run the manual-override cell above. "
    "Locally: create a .env in the repo root with MLFLOW_TRACKING_USERNAME=... and "
    "MLFLOW_TRACKING_PASSWORD=... (inference cannot read the Model Registry without it)."
)
print("MLflow credentials loaded from:", creds_source)

MLflow credentials loaded from: pre-set environment variables


##  MLflow tracking store and Model Registry coordinates

Shared DagsHub MLflow server - the single source of truth for cross-model WMAE comparison
and the Model Registry. Does not silently fall back to a local store if auth fails: a local
`./mlruns` contains none of the team's runs, so "falling back" would just mean silently
failing to find the model.

### On stage vs alias

MLflow **3.x removed model stages** (`Staging` / `Production` / `Archived`) in favour of
**aliases**. `MODEL_ALIAS = "champion"` below is the direct replacement for what used to be
the `Production` stage, and it resolves through the URI `models:/<name>@champion`. If you are
pinned to MLflow 2.x and need the old stage semantics, set
`MODEL_URI = f"models:/{MODEL_NAME}/Production"` instead.

In [5]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# --- Model Registry coordinates: the winning model, by Kaggle score ---
# The registry entry is named for its ROLE, not its algorithm: whichever model wins, inference
# always fetches "Best_Model". Swapping in a different winner later means pointing the alias at
# a new version, with no edit to this notebook.
MODEL_NAME = "Best_Model"
MODEL_ALIAS = "champion"  # MLflow 3.x replacement for the old "Production" stage
MODEL_URI = f"models:/{MODEL_NAME}@{MODEL_ALIAS}"

# The run the registered model is sourced from, if the registry still needs bootstrapping.
# This stays "Prophet_Final" on purpose - it is the provenance (which run produced these
# weights), not the role. Best_Model v1 points back at it.
SOURCE_EXPERIMENT = "Prophet_Training"
SOURCE_RUN_NAME = "Prophet_Final"

try:
    mlflow.set_experiment(SOURCE_EXPERIMENT)
    MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and MLFLOW_TRACKING_PASSWORD "
        "(a DagsHub access token) as environment variables, then re-run this cell. A 401 here "
        "usually means the token has expired - mint a fresh one at "
        "https://dagshub.com/user/settings/tokens. Not falling back to a local ./mlruns: it "
        "holds none of the team's runs, so the registry lookup below would fail anyway."
    ) from e

client = MlflowClient()
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Model URI:          ", MODEL_URI)

MLflow tracking URI: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow
Model URI:           models:/Best_Model@champion


##  Load the raw, un-preprocessed test set

Straight off disk, exactly as Kaggle ships it: `Store, Dept, Date, IsHoliday`. **No merge, no
cleaning, no feature engineering** - all of that is the loaded pipeline's job.

We deliberately do not call `src.preprocessing.load_raw_data()` here: it reads all four tables
(train, test, features, stores) and forward-fills the macro columns, and inference needs none
of that. It does hand us one useful detail though - unzipped Kaggle artifacts leave `test.csv`
as a *directory* containing a same-named CSV - so we accept either layout.

In [6]:
def resolve_raw_csv(name: str) -> Path:
    """Kaggle zip artifacts unpack `test.csv` as a directory holding `test.csv`.

    Same convention `src/preprocessing.py`'s `_NESTED_FILES` handles. Accept either layout.
    """
    flat = DATA_DIR / name
    nested = DATA_DIR / name / name
    if nested.is_file():
        return nested
    if flat.is_file():
        return flat
    raise FileNotFoundError(
        f"{name} not found at {flat} or {nested}. data/raw/ is gitignored, so a fresh clone "
        "starts with an empty data/raw/. On Colab the init cell copies it down from Drive. "
        "Locally, download the competition data from "
        "https://www.kaggle.com/c/walmart-recruiting-store-sales-forecasting/data and unzip "
        f"it into {DATA_DIR}."
    )


raw_test = pd.read_csv(resolve_raw_csv("test.csv"), parse_dates=["Date"])

print("raw test shape:", raw_test.shape)
print("columns:       ", list(raw_test.columns))
print("date span:     ", raw_test["Date"].min().date(), "->", raw_test["Date"].max().date())
print("NaNs per column:")
print(raw_test.isna().sum().to_string())

assert len(raw_test) == EXPECTED_ROWS, f"expected {EXPECTED_ROWS} test rows, got {len(raw_test)}"
raw_test.head()

raw test shape: (115064, 4)
columns:        ['Store', 'Dept', 'Date', 'IsHoliday']
date span:      2012-11-02 -> 2013-07-26
NaNs per column:
Store        0
Dept         0
Date         0
IsHoliday    0


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


##  Bootstrap the Model Registry (run once)

**Read this before running.** As of writing, the Model Registry is *empty*: no notebook in
this repo has ever called `mlflow.register_model` or passed `registered_model_name`.
`model_experiment_prophet.ipynb` persists the winner with `joblib.dump(...)` +
`mlflow.log_artifact(...)`, which attaches a **plain file** to the `Prophet_Final` run. A plain
file is not an MLflow *model*, so there is nothing for a `models:/...` URI to resolve and
nothing to attach a stage/alias to.

This cell closes that gap **without refitting anything** (a full Prophet panel refit is tens of
minutes). It takes the pipeline that already exists, wraps it in an `mlflow.pyfunc` model so the
registry has a loadable flavour to serve, registers it as `Best_Model`, and points the
`champion` alias at that version.

Note the two names do different jobs, and both are worth keeping. The **run** stays
`Prophet_Final` - that is provenance, the run whose metrics and params justified this choice.
The **registered model** is `Best_Model` - that is a role, so when a different model wins later,
you register it as a new version of `Best_Model` and move the alias, and nothing downstream of
here changes.

The wrapper exists for exactly one reason: to give the registry a flavour whose `predict()` is
`ProphetPanelPipeline.predict()` - raw test frame in, `np.ndarray` out. `code_paths` ships `src/`
inside the model artifact, and `load_context` re-aliases it to the `src.`-prefixed module path the
pickle refers to, so the registered model also loads on a machine without this repo checked out.

It is **idempotent and guarded**: if the alias already resolves, it does nothing. Set
`FORCE_REREGISTER = True` to push a new version anyway.

In [7]:
FORCE_REREGISTER = False


def registry_has_champion() -> bool:
    try:
        client.get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
        return True
    except Exception:
        return False


class ProphetPanelWrapper(mlflow.pyfunc.PythonModel):
    """pyfunc flavour around the already-fitted ProphetPanelPipeline.

    predict() forwards the raw test frame straight through, so the registered model's contract
    is identical to the pipeline's: raw test.csv-shaped DataFrame -> np.ndarray.
    """

    def load_context(self, context):
        import sys
        import types

        import joblib

        # The pickle names the class `src.prophet_forecaster.ProphetPanelPipeline`. When this
        # model is served outside the repo, code_paths gives us a top-level
        # `prophet_forecaster` module instead, so alias it under `src.` before unpickling.
        try:
            import src.prophet_forecaster  # noqa: F401
        except ModuleNotFoundError:
            import prophet_forecaster

            pkg = types.ModuleType("src")
            pkg.__path__ = []
            sys.modules.setdefault("src", pkg)
            sys.modules["src.prophet_forecaster"] = prophet_forecaster

        self.pipeline = joblib.load(context.artifacts["pipeline"])

    def predict(self, context, model_input, params=None):
        return self.pipeline.predict(model_input)


def find_source_run():
    """The Prophet_Final run whose artifact we are promoting into the registry."""
    exp = client.get_experiment_by_name(SOURCE_EXPERIMENT)
    if exp is None:
        return None
    runs = client.search_runs(
        [exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{SOURCE_RUN_NAME}'",
        order_by=["start_time DESC"],
        max_results=1,
    )
    return runs[0] if runs else None


def local_or_downloaded_joblib(run) -> Path:
    """Prefer the local joblib the Prophet notebook wrote; otherwise pull it off the run."""
    local = SUBMISSIONS_DIR / "_prophet_forecaster.joblib"
    if local.is_file():
        print(f"using local pipeline artifact: {local} ({local.stat().st_size / 1e6:.1f} MB)")
        return local
    if run is None:
        raise FileNotFoundError(
            f"No local {local}, and no {SOURCE_RUN_NAME} run on the tracking server. Re-run "
            "model_experiment_prophet.ipynb to produce the fitted pipeline."
        )
    print(f"downloading _prophet_forecaster.joblib from run {run.info.run_id} ...")
    return Path(
        mlflow.artifacts.download_artifacts(
            run_id=run.info.run_id, artifact_path="_prophet_forecaster.joblib"
        )
    )


if registry_has_champion() and not FORCE_REREGISTER:
    _mv = client.get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
    print(f"Registry already serves {MODEL_URI} (version {_mv.version}). Nothing to do.")
else:
    from mlflow.models import infer_signature

    source_run = find_source_run()
    pipeline_path = local_or_downloaded_joblib(source_run)

    # Load once locally to build an honest signature from a real prediction.
    local_pipeline = joblib.load(pipeline_path)
    sample_in = raw_test.head(5)
    signature = infer_signature(sample_in, local_pipeline.predict(sample_in))

    # Log the model INTO the existing Prophet_Final run when we can find it, so the registered
    # version stays attached to the metrics and params that justified picking it.
    run_kwargs = (
        {"run_id": source_run.info.run_id}
        if source_run is not None
        else {"run_name": "Prophet_Final_Registry_Bootstrap"}
    )
    with mlflow.start_run(**run_kwargs):
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=ProphetPanelWrapper(),
            artifacts={"pipeline": str(pipeline_path)},
            code_paths=[str(SRC_DIR)],
            signature=signature,
            input_example=sample_in,
            registered_model_name=MODEL_NAME,
        )

    latest = max(
        client.search_model_versions(f"name='{MODEL_NAME}'"),
        key=lambda v: int(v.version),
    )
    client.set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, latest.version)
    print(f"Registered {MODEL_NAME} v{latest.version}, alias @{MODEL_ALIAS} now points at it.")

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


using local pipeline artifact: /content/walmart-sales-forecasting/submissions/_prophet_forecaster.joblib (20.9 MB)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    3.1s
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/12 17:36:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
202

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    2.4s
Successfully registered model 'Best_Model'.
2026/07/12 17:36:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best_Model, version 1
Created version '1' of model 'Best_Model'.


🏃 View run Prophet_Final at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/7/runs/50a88f1ef48d4ba9b34312e16afaac86
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/7
Registered Best_Model v1, alias @champion now points at it.


##  Fetch the pipeline from the Model Registry

The source of truth for "what model is in production" is the registry alias, not a file path on
someone's laptop. `mlflow.pyfunc.load_model` pulls the exact version `champion` points at,
together with the `src/` code it was logged with.

No local-file fallback, on purpose: if this fails, the right fix is to bootstrap or repoint the
registry above, not to quietly score with whatever happens to be lying around in `submissions/`.

In [8]:
mv = client.get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
source_metrics = client.get_run(mv.run_id).data.metrics

print(f"loading {MODEL_URI}")
print(f"  version:  {mv.version}")
print(f"  from run: {mv.run_id}")
print("  metrics: ", {k: round(v, 2) for k, v in source_metrics.items()})

pipeline = mlflow.pyfunc.load_model(MODEL_URI)
print("\nloaded:", type(pipeline).__name__)

loading models:/Best_Model@champion
  version:  1
  from run: 50a88f1ef48d4ba9b34312e16afaac86
  metrics:  {'wmae_testlike_39w': 1962.57, 'reference_prophet_representative_series_wmae_mean': 12951.05, 'mae': 1548.92, 'reference_prophet_aggregate_wmae': 730720.62, 'wmae': 1565.07, 'mae_testlike_39w': 1940.83}



loaded: PyFuncModel


##  Predict: raw test in, predictions out

One call. The raw frame goes in untouched; preprocessing, per-series model selection, the
department-seasonal fallback for series with no fitted model, and the non-negativity clip all
happen inside the loaded pipeline.

**This is slow, and it is CPU-bound.** It deserializes ~3,000 Prophet models from JSON and
forecasts each one, fanned out across cores by joblib. Expect minutes, not seconds, and expect
the cell to look idle while it works.

In [9]:
import time

_t0 = time.time()
predictions = pipeline.predict(raw_test)
print(f"predict done in {(time.time() - _t0) / 60:.1f} min")

predictions = np.asarray(predictions, dtype=float).ravel()
print("predictions:", predictions.shape)
print(pd.Series(predictions).describe().to_string())

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  20 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done  68 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done 148 tasks      | elapsed:    9.2s
[Parallel(n_jobs=-1)]: Done 228 tasks      | elapsed:   15.5s
[Parallel(n_jobs=-1)]: Done 340 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 452 tasks      | elapsed:   30.3s
[Parallel(n_jobs=-1)]: Done 596 tasks      | elapsed:   37.6s
[Parallel(n_jobs=-1)]: Done 740 tasks      | elapsed:   48.0s
[Parallel(n_jobs=-1)]: Done 916 tasks      | elapsed:   59.3s
[Parallel(n_jobs=-1)]: Done 1092 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 1300 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 1508 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 1748 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 1988 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 2260 tasks      | elaps

predict done in 3.3 min
predictions: (115064,)
count    115064.000000
mean      16602.154889
std       23991.999627
min           0.000000
25%        2103.881829
50%        7656.056559
75%       20949.551146
max      652215.256961


##  Format, sanity-check, and write the submission

The Walmart Recruiting format is exactly two columns:

- `Id` - `{Store}_{Dept}_{YYYY-MM-DD}`, built from the raw test rows in their original order
- `Weekly_Sales` - the forecast

The checks below are hard assertions rather than prints. A submission that is silently the wrong
shape, or silently full of NaNs, is worse than one that fails loudly right here.

In [10]:
submission = pd.DataFrame(
    {
        "Id": (
            raw_test["Store"].astype(str)
            + "_"
            + raw_test["Dept"].astype(str)
            + "_"
            + raw_test["Date"].dt.strftime("%Y-%m-%d")
        ),
        "Weekly_Sales": predictions,
    }
)

# --- sanity checks ---
assert list(submission.columns) == ["Id", "Weekly_Sales"], submission.columns
assert submission.shape == (EXPECTED_ROWS, 2), submission.shape
assert not submission["Weekly_Sales"].isna().any(), "NaN predictions in the submission"
assert np.isfinite(submission["Weekly_Sales"]).all(), "non-finite predictions in the submission"
assert submission["Id"].is_unique, "duplicate Ids in the submission"

# Optional cross-check against Kaggle's own sampleSubmission when it is present locally: the
# strongest available check is that our Id set IS their Id set.
sample_path = DATA_DIR / "sampleSubmission.csv"
if sample_path.is_file():
    sample_ids = pd.read_csv(sample_path, usecols=["Id"])["Id"]
    assert set(submission["Id"]) == set(sample_ids), "Id set does not match sampleSubmission.csv"
    print("Id set matches sampleSubmission.csv exactly")
else:
    print(f"(no {sample_path.name} on disk - skipped the Id set cross-check)")

print("rows:      ", len(submission))
print("NaNs:      ", int(submission["Weekly_Sales"].isna().sum()))
print("negatives: ", int((submission["Weekly_Sales"] < 0).sum()))
print(
    "Weekly_Sales min/mean/max:",
    f"{submission['Weekly_Sales'].min():,.2f} /",
    f"{submission['Weekly_Sales'].mean():,.2f} /",
    f"{submission['Weekly_Sales'].max():,.2f}",
)

submission.to_csv(SUBMISSION_PATH, index=False)
print("\nwrote", SUBMISSION_PATH)
submission.head()

Id set matches sampleSubmission.csv exactly
rows:       115064
NaNs:       0
negatives:  0
Weekly_Sales min/mean/max: 0.00 / 16,602.15 / 652,215.26

wrote /content/walmart-sales-forecasting/submissions/final_submission.csv


,Id,Weekly_Sales
0,1_1_2012-11-02,25206.189886
1,1_1_2012-11-09,24255.637341
2,1_1_2012-11-16,24659.220328
3,1_1_2012-11-23,25824.241655
4,1_1_2012-11-30,28127.713635


##  Done

`submissions/final_submission.csv` is ready to upload to Kaggle. It was produced by fetching
`models:/Best_Model@champion` from the Model Registry and calling `.predict()` once on the raw,
untouched `data/raw/test.csv`.

### Regression check against the known-good submission

This file should come out near-identical to `submissions/prophet_submission.csv`, which
`model_experiment_prophet.ipynb` wrote directly from the same fitted pipeline. If the two diverge
materially, the registry is serving a different model than the one that earned the leaderboard
score - investigate before submitting.

In [11]:
prev_path = SUBMISSIONS_DIR / "prophet_submission.csv"
if prev_path.is_file():
    prev = pd.read_csv(prev_path).set_index("Id")["Weekly_Sales"]
    curr = submission.set_index("Id")["Weekly_Sales"]
    assert set(curr.index) == set(prev.index), "Id sets differ from prophet_submission.csv"
    delta = (curr - prev.reindex(curr.index)).abs()
    print("absolute difference vs prophet_submission.csv:")
    print(delta.describe().to_string())
    print(f"\nmax abs diff: {delta.max():,.4f}")
else:
    print(f"(no {prev_path.name} on disk - skipped the regression check)")

absolute difference vs prophet_submission.csv:
count    1.150640e+05
mean     3.835800e-13
std      1.651987e-12
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.820766e-11

max abs diff: 0.0000
